In [16]:
#Import Libraries
import pandas as pd
import numpy as np
from mrmr import mrmr_classif

from sklearn.model_selection import KFold, GridSearchCV
# from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
print("Libraries imported successfulldy.")


Libraries imported successfulldy.


In [10]:
# LOAD DATA
df = pd.read_csv("data/1/train.csv")

X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# CHECK MISSING VALUES
print("Missing values per column:\n")
print(X.isnull().sum())

Missing values per column:

F1       0
F2     613
F3       0
F4       0
F5       0
F6       0
F7       0
F8       0
F9       0
F10      0
F11      0
F12      0
F13      0
F14      0
F15      0
F16      0
F17      0
F18      0
F20      0
dtype: int64


In [ ]:
print("Columns:" + str(X.columns.tolist()))
print("Number of duplicated rows: " + str(df.duplicated().sum()))

Columns:['F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8', 'F9', 'F10', 'F11', 'F12', 'F13', 'F14', 'F15', 'F16', 'F17', 'F18', 'F20']
Number of duplicated rows: 125


In [12]:
# HANDLE MISSING VALUES
# -----------------------------

# OPTION 1: Fill with global mean (simple)
# X = X.fillna(X.mean())

# OPTION 2 (Better): Fill using class-wise mean

# for col in X.columns:
#     if X[col].isnull().sum() > 0:
#         X[col] = X.groupby(y)[col].transform(lambda x: x.fillna(x.mean()))

print("Missing values per column:\n")
print(X.isnull().sum())


Missing values per column:

F1       0
F2     613
F3       0
F4       0
F5       0
F6       0
F7       0
F8       0
F9       0
F10      0
F11      0
F12      0
F13      0
F14      0
F15      0
F16      0
F17      0
F18      0
F20      0
dtype: int64


In [13]:
# MODELS + PARAMS
models = {
    "SVM": (
        SVC(),
        {
            "C": [1, 10],
            "kernel": ["rbf"],
            "gamma": ["scale"]
        }
    ),
    
    "KNN": (
        KNeighborsClassifier(),
        {
            "n_neighbors": [5, 7]
        }
    ),
    
    "DecisionTree": (
        DecisionTreeClassifier(),
        {
            "max_depth": [None, 10]
        }
    ),
    
    "RandomForest": (
        RandomForestClassifier(),
        {
            "n_estimators": [100],
            "max_depth": [None, 10]
        }
    ),
    
    "MLP": (
        MLPClassifier(max_iter=500),
        {
            "hidden_layer_sizes": [(100,)],
            "learning_rate_init": [0.001]
        }
    )
}

In [14]:
# MRMR PARAMS
k_features = 10  # number of top features to select

In [18]:
# -----------------------------
# 10-FOLD STRATIFIED CV
# -----------------------------
# kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
kf = KFold(n_splits=10, shuffle=True, random_state=42)
results = {name: {"acc": [], "f1": [], "best_params": []} for name in models.keys()}

for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    print(f"\nFold {fold}")
    
    X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # -----------------------------
    # MISSING VALUE IMPUTATION (inside fold)
    # -----------------------------
    for col in X_train.columns:
        mean_val = X_train[col].mean()
        X_train[col] = X_train[col].fillna(mean_val)
        X_val[col] = X_val[col].fillna(mean_val)

    # -----------------------------
    # SCALE DATA (inside fold)
    # -----------------------------
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
    X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns)

    # -----------------------------
    # MRMR FEATURE SELECTION
    # -----------------------------
    # Reset indices to avoid alignment issues
    X_train_scaled = X_train_scaled.reset_index(drop=True)
    X_val_scaled = X_val_scaled.reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)
    y_val = y_val.reset_index(drop=True)

    top_features = mrmr_classif(X=X_train_scaled, y=y_train, K=k_features)

    X_train_selected = X_train_scaled[top_features]
    X_val_selected = X_val_scaled[top_features]

    # -----------------------------
    # MRMR FEATURE SELECTION
    # -----------------------------
    top_features = mrmr_classif(X=X_train_scaled, y=y_train, K=10)
    X_train_selected = X_train_scaled[top_features]
    X_val_selected = X_val_scaled[top_features]

    # -----------------------------
    # TRAIN MODELS WITH NESTED CV
    # -----------------------------
    for name, (model, params) in models.items():
        grid = GridSearchCV(model, params, cv=3, scoring='f1_macro')
        grid.fit(X_train_selected, y_train)

        best_model = grid.best_estimator_
        y_pred = best_model.predict(X_val_selected)

        acc = accuracy_score(y_val, y_pred)
        f1 = f1_score(y_val, y_pred, average='macro')

        results[name]["acc"].append(acc)
        results[name]["f1"].append(f1)
        results[name]["best_params"].append(grid.best_params_)

        print(f"{name} | Acc: {acc:.4f} | F1: {f1:.4f} | Best Params: {grid.best_params_}")



Fold 1


100%|██████████| 10/10 [00:00<00:00, 41.55it/s]


SVM | Acc: 0.9531 | F1: 0.9403 | Best Params: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
KNN | Acc: 0.9645 | F1: 0.9362 | Best Params: {'n_neighbors': 5}
DecisionTree | Acc: 1.0000 | F1: 1.0000 | Best Params: {'max_depth': None}
RandomForest | Acc: 1.0000 | F1: 1.0000 | Best Params: {'max_depth': None, 'n_estimators': 100}
MLP | Acc: 0.9802 | F1: 0.9745 | Best Params: {'hidden_layer_sizes': (100,), 'learning_rate_init': 0.001}

Fold 2


100%|██████████| 10/10 [00:00<00:00, 43.34it/s]


SVM | Acc: 0.9645 | F1: 0.9523 | Best Params: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
KNN | Acc: 0.9791 | F1: 0.7122 | Best Params: {'n_neighbors': 5}
DecisionTree | Acc: 1.0000 | F1: 1.0000 | Best Params: {'max_depth': None}
RandomForest | Acc: 1.0000 | F1: 1.0000 | Best Params: {'max_depth': None, 'n_estimators': 100}
MLP | Acc: 0.9917 | F1: 0.9879 | Best Params: {'hidden_layer_sizes': (100,), 'learning_rate_init': 0.001}

Fold 3


...

SVM | Acc: 0.9593 | F1: 0.9376 | Best Params: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
KNN | Acc: 0.9802 | F1: 0.9509 | Best Params: {'n_neighbors': 5}
DecisionTree | Acc: 1.0000 | F1: 1.0000 | Best Params: {'max_depth': None}
RandomForest | Acc: 0.9990 | F1: 0.9988 | Best Params: {'max_depth': None, 'n_estimators': 100}
MLP | Acc: 0.9896 | F1: 0.9812 | Best Params: {'hidden_layer_sizes': (100,), 'learning_rate_init': 0.001}

Fold 9


100%|██████████| 10/10 [00:00<00:00, 42.72it/s]


SVM | Acc: 0.9770 | F1: 0.9708 | Best Params: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
KNN | Acc: 0.9864 | F1: 0.9787 | Best Params: {'n_neighbors': 5}
DecisionTree | Acc: 1.0000 | F1: 1.0000 | Best Params: {'max_depth': None}
RandomForest | Acc: 1.0000 | F1: 1.0000 | Best Params: {'max_depth': None, 'n_estimators': 100}
MLP | Acc: 0.9969 | F1: 0.9960 | Best Params: {'hidden_layer_sizes': (100,), 'learning_rate_init': 0.001}

Fold 10


100%|██████████| 10/10 [00:00<00:00, 56.23it/s]


SVM | Acc: 0.9739 | F1: 0.9727 | Best Params: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
KNN | Acc: 0.9885 | F1: 0.7220 | Best Params: {'n_neighbors': 5}
DecisionTree | Acc: 1.0000 | F1: 1.0000 | Best Params: {'max_depth': None}
RandomForest | Acc: 1.0000 | F1: 1.0000 | Best Params: {'max_depth': None, 'n_estimators': 100}
MLP | Acc: 0.9896 | F1: 0.9890 | Best Params: {'hidden_layer_sizes': (100,), 'learning_rate_init': 0.001}


In [ ]:
# FINAL RESULTS (Mean ± Std)
print("\n================ FINAL RESULTS (MRMR) ================\n")
for name in models.keys():
    acc_mean = np.mean(results[name]["acc"])
    acc_std = np.std(results[name]["acc"])
    f1_mean = np.mean(results[name]["f1"])
    f1_std = np.std(results[name]["f1"])

    print(f"{name}:")
    print(f"  Accuracy: {acc_mean:.4f} ± {acc_std:.4f}")
    print(f"  F1 Score: {f1_mean:.4f} ± {f1_std:.4f}")
    print("--------------------------------------------------")

RESULTS (Mean ± Std)
SVM:
  Accuracy: 0.9730 ± 0.0060
  F1 Score: 0.8786 ± 0.1107
KNN:
  Accuracy: 0.9789 ± 0.0056
  F1 Score: 0.8380 ± 0.1201
DecisionTree:
  Accuracy: 1.0000 ± 0.0000
  F1 Score: 1.0000 ± 0.0000
RandomForest:
  Accuracy: 0.9995 ± 0.0010
  F1 Score: 0.9994 ± 0.0012
MLP:
  Accuracy: 0.9867 ± 0.0054
  F1 Score: 0.9560 ± 0.0759
